# Importing required libraries

In [ ]:
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

: 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Loading & Reading Dataset

In [ ]:
train = pd.read_csv("./bike-sharing-demand/train.csv")
test = pd.read_csv("./bike-sharing-demand/test.csv")

In [ ]:
train.shape

In [ ]:
test.shape

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
df = pd.concat([train, test], axis=0, ignore_index=True)

# EDA

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()


The 6,493 missing values (NaN) in the **count**, **casual**, and **registered** columns of the combined dataset do not indicate data quality problems. Instead, they originate from the **Test Set** partition, where the target values are intentionally absent. These missing values correspond to the outputs that the model is required to predict using the independent variables (features) that were not available during training. As a result, these entries are meant to be filled with the model's predictions rather than through statistical imputation.


# Adding Features

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.dayofweek
df['dayofweek'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['year'] = df['datetime'].dt.year

In [ ]:
df.drop(['datetime', 'casual', 'registered'], axis=1, inplace=True)

In [ ]:
df['temp_diff'] = abs(df['temp'] - df['atemp'])

In [ ]:
df['is_peak'] = ((df['workingday'] == 1) & (df['hour'].isin([8, 17, 18]))).astype(int)

In [ ]:
df = pd.get_dummies(df, columns=['season', 'weather'], drop_first=True)

In [ ]:
df.rename(columns={'season_2': 'summer','season_3': 'fall','season_4': 'winter','weather_2': 'cloudy','weather_3': 'rainy', 'weather_4': 'heavy_storm'}, inplace=True)
df.rename(columns={'workingday': 'is_workingday','holiday': 'is_holiday'}, inplace=True)

In [ ]:
df.columns

In [ ]:
df.tail()

In [ ]:
train = df[df['count'].notnull()].copy()
test = df[df['count'].isnull()].drop(['count'], axis=1)

In [ ]:
sns.barplot(data=train, x='dayofweek', y='count', hue='year', palette='Set1').set(xticklabels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],title='Weekly Rental Patterns: 2011 vs 2012');

This chart demonstrates that the bike-sharing system nearly doubled its adoption and capacity across all days of the week between 2011 and 2012. Examining the weekly pattern, weekday (Monday–Friday) rentals remain relatively consistent, reflecting a stable commuting demand. However, the most substantial increase is observed on weekends (Saturday and Sunday), indicating that the service is widely used for recreational activities in addition to everyday transportation.

In [ ]:
train['Weather'] = train['rainy'] + (train['heavy_storm'] * 2)
sns.barplot(data=train, x='Weather', y='count', palette=['gold', 'skyblue', 'darkred']).set(xticklabels=['Sunny', 'Rainy', 'Stormy'], title='Rental Demand by Weather Condition');

In [ ]:
test['Weather'] = test['rainy'] + (test['heavy_storm'] * 2)

In [ ]:
px.bar(train.groupby(['hour', 'is_workingday'])['count'].mean().reset_index(), x="hour", y="count", color="is_workingday", barmode="group",
title="Hourly Peaks: Working Day vs Weekend").show()

The chart shows that bicycle rental demand is influenced not only by the time of day but also by human behavior across different days of the week. On working days, two prominent peaks occur at 8 AM and 5 PM, corresponding to commuters travelling to and from work. In contrast, weekends lack the morning commute peak, with demand increasing gradually throughout the afternoon as people use the service for leisure activities. This indicates that, for the model to make accurate predictions, it must consider both the hour of the day and whether that hour falls on a working day or a holiday, as these factors together determine the expected number of rentals.

In [ ]:
px.scatter(train, x="temp", y="humidity", color="count", size="count", title="Temp & Humidity vs Count").show()

This chart identifies the optimal conditions, or "comfort zone," for bicycle rentals, where the largest and brightest clusters—representing the highest rental counts—are concentrated at temperatures between **20°C and 30°C** and moderate humidity levels ranging from **20% to 60%**. As weather conditions become more extreme, such as near-freezing temperatures or humidity levels approaching **90%**, the points become smaller and darker, indicating a decline in rental demand. This pattern demonstrates that people are less likely to ride in harsh weather conditions. Therefore, the model must evaluate the combined effects of temperature and humidity to accurately predict periods of high bicycle rental demand.


In [ ]:
plt.figure(figsize=(15, 6))
sns.barplot(data=train, x='month', y='count', hue='is_peak', palette='magma').set_title("Monthly Distribution: Peak Hours (1) vs Normal Hours (0)"); plt.show()

This chart illustrates how the influence of **peak hours** varies across different months, demonstrating that the **is_peak** feature consistently generates higher rental demand than normal hours throughout the year. The pronounced difference between the pink bars (peak hours) and the purple bars (normal hours) indicates that although overall demand declines during winter and increases in summer, peak-hour activity remains the primary factor driving bicycle rentals. Consequently, the model must account for the fact that a peak hour in **July** has a substantially greater impact than a peak hour in **January** to accurately predict the 6,493 missing rental values.


# Data Processing

In [ ]:
x_train = train.drop(['count'], axis=1)
y_train = train['count']

# Data Modeling

In [ ]:
model = RandomForestRegressor(n_estimators=400,max_depth=50,n_jobs=-2,random_state=52)

In [ ]:
model.fit(x_train, y_train)
train_predictions = model.predict(x_train)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
mae = mean_absolute_error(y_train, train_predictions)
r2 = model.score(x_train, y_train)
print(f"R2 Skoru (Train): {r2:.2f}")
print(f"RMSE (Train): {rmse:.2f}")
print(f"MAE (Train): {mae:.2f}")

### Training Run Summary & Visualizations
This section creates a summary table for the model run and plots evaluation metrics. The repository does not contain a Kaggle score, so the Kaggle score plot shows its missing status.


In [ ]:
run_summary = pd.DataFrame([
    {
        "run": 1,
        "model": "Random Forest Regressor",
        "n_estimators": 400,
        "max_depth": 50,
        "random_state": 52,
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
        "kaggle_score": None,
    }
])

run_summary_display = run_summary.copy()
run_summary_display["kaggle_score"] = run_summary_display["kaggle_score"].fillna("Not available")

display(run_summary_display)

metrics = run_summary.melt(id_vars=["run"], value_vars=["r2", "rmse", "mae"], var_name="metric", value_name="value")
metric_fig = px.bar(
    metrics,
    x="metric",
    y="value",
    color="metric",
    title="Training Run Evaluation Metrics",
    labels={"value": "Metric Value", "metric": "Metric"},
)
metric_fig.update_layout(showlegend=False)
metric_fig.show()

submission_files = [
    ("submission.csv", "Main"),
    ("submission (1).csv", "Submission 1"),
    ("submission (2).csv", "Submission 2"),
    ("submission (3).csv", "Submission 3"),
    ("submission (4).csv", "Submission 4"),
]

submission_stats = []
for file_path, name in submission_files:
    try:
        sub_df = pd.read_csv(file_path)
        submission_stats.append({
            "submission": name,
            "mean_count": sub_df["count"].mean(),
            "min_count": sub_df["count"].min(),
            "max_count": sub_df["count"].max(),
            "std_count": sub_df["count"].std(),
        })
    except FileNotFoundError:
        print(f"Warning: {file_path} not found")

submission_comparison = pd.DataFrame(submission_stats)
print("\nSubmission Statistics:")
display(submission_comparison)

submission_melted = submission_comparison.melt(
    id_vars=["submission"],
    value_vars=["mean_count", "min_count", "max_count", "std_count"],
    var_name="statistic",
    value_name="value"
)

submission_fig = px.bar(
    submission_melted,
    x="submission",
    y="value",
    color="statistic",
    barmode="group",
    title="Submission Comparison: Prediction Statistics",
    labels={"value": "Count Value", "submission": "Submission File"},
)
submission_fig.show()

kaggle_fig = px.bar(
    run_summary.assign(kaggle_score_value=run_summary["kaggle_score"].fillna(0)),
    x="run",
    y="kaggle_score_value",
    title="Kaggle Score Status",
    labels={"kaggle_score_value": "Kaggle Score", "run": "Run"},
)
kaggle_fig.add_annotation(
    x=1,
    y=0.1,
    text="Kaggle score not available in repository",
    showarrow=False,
    yanchor="bottom",
    xanchor="center",
)
kaggle_fig.update_yaxes(range=[0, 1], visible=False)
kaggle_fig.update_traces(marker_color="indianred")
kaggle_fig.show()


### Driving Predictions from the Processed Data

In [ ]:
pred_test = model.predict(test)
pred_test = np.maximum(1, pred_test)

In [ ]:
test_orijinal = pd.read_csv("./bike-sharing-demand/test.csv")
submission = pd.DataFrame({"datetime": test_orijinal["datetime"],"count": pred_test})
submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)

In [ ]:
joblib.dump(model, 'bike_model.pkl')

## Conclusion

Our model has successfully captured the underlying dynamics of the bike-sharing system, achieving an **R² score of 0.99** and a **Mean Absolute Error (MAE) of 9.37** on the training data. The analysis demonstrates that bicycle rental demand is influenced not only by time but also by human behavior and weather conditions. The consistent importance of the **is_peak** feature across all seasons indicates that the model effectively accounts for seasonal variations while recognizing peak-hour demand. It accurately distinguishes the pronounced weekday commuting peaks at **8 AM** and **5 PM** from the gradual increase in weekend afternoon rentals driven by leisure activities. In addition, the near doubling of bike-sharing adoption between **2011 and 2012**, along with the identified weather "comfort zone" of **20°C to 30°C**, explains the model's strong predictive performance, as reflected by its **Root Mean Squared Error (RMSE) of 15.03**.